# AegisDesk Kaggle DPO Notebook

This notebook is the fuller Kaggle fallback path for AegisDesk when GRPO keeps collapsing into identical reward groups.

It bootstraps the base preference dataset when a fresh clone does not include it, optionally mixes in harvested AegisDesk pairs, previews the resulting corpus, and then launches a QLoRA DPO run.

**Use this when:**
- the GRPO notebook keeps logging identical reward groups
- advantage stays near zero after the core-task probe
- you want a practical Kaggle path instead of more reward-shaping work


In [ ]:
%%capture
!pip install -U unsloth trl transformers datasets peft accelerate bitsandbytes sentencepiece protobuf

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO_DIR = Path('/kaggle/working/AegisDesk')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/kumarabhik/AegisDesk.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR), '--quiet'], check=True)
os.chdir(REPO_DIR)

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ.setdefault('WANDB_DISABLED', 'true')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

MODEL_NAME = os.environ.get('AEGIS_DPO_MODEL_NAME', 'Qwen/Qwen2.5-1.5B-Instruct')
BASE_PREF_PATH = 'training/data/support_pref.jsonl'
DATASET_PATH = BASE_PREF_PATH
PROJECT_PREF_PATH = 'training/data/aegisdesk_pref.jsonl'
OUTPUT_DIR = '/kaggle/working/aegisdesk-dpo'

print('Repo:', REPO_DIR)
print('Model:', MODEL_NAME)
print('Base dataset:', BASE_PREF_PATH)
print('Project mix output:', PROJECT_PREF_PATH)
print('Output:', OUTPUT_DIR)
print('WANDB_DISABLED:', os.environ.get('WANDB_DISABLED'))


## Bootstrap Preference Data and Build the Project Mix

Fresh GitHub clones do not include the large generated preference corpus. This cell regenerates it when needed, then tries to mix in harvested AegisDesk preference pairs if they exist.


In [ ]:
base_pref = Path(BASE_PREF_PATH)
if not base_pref.exists():
    print('support_pref.jsonl is missing in a fresh clone. Building it now...')
    subprocess.run([sys.executable, 'scripts/fetch_real_datasets.py'], check=True)
    print('Base preference dataset ready:', base_pref)

build_cmd = [
    sys.executable,
    '-m', 'training.build_aegisdesk_preference_corpus',
    '--output', PROJECT_PREF_PATH,
    '--upsample-project-pairs', '4',
]
result = subprocess.run(build_cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print('Project-specific preference build failed; falling back to the base preference dataset.')
    print(result.stderr)
    DATASET_PATH = BASE_PREF_PATH
    print('Using base dataset:', DATASET_PATH)
else:
    DATASET_PATH = PROJECT_PREF_PATH
    print('Using project-mixed dataset:', DATASET_PATH)


In [ ]:
import json
from collections import Counter
from itertools import islice

dataset_path = Path(DATASET_PATH)
assert dataset_path.exists(), f"Missing dataset: {dataset_path}"

rows = []
with dataset_path.open(encoding='utf-8') as handle:
    for line in islice(handle, 1000):
        line = line.strip()
        if line:
            rows.append(json.loads(line))

print('Preview dataset:', dataset_path)
print('Sampled rows for sanity check:', len(rows))
if rows:
    print(rows[0].keys())
    print(rows[0]['prompt'][:300])
    print(rows[0]['chosen'][:300])
    source_counts = Counter(str(row.get('source', 'unknown')) for row in rows)
    print('Top sources in sample:', source_counts.most_common(5))
    gaps = []
    for row in rows:
        if row.get('chosen_score') is not None and row.get('rejected_score') is not None:
            try:
                gaps.append(float(row['chosen_score']) - float(row['rejected_score']))
            except (TypeError, ValueError):
                pass
    if gaps:
        print('Score-gap sample stats:', {'min': min(gaps), 'max': max(gaps), 'mean': round(sum(gaps) / len(gaps), 4)})


In [ ]:
train_cmd = [
    sys.executable,
    '-m', 'training.train_unsloth_dpo',
    '--dataset', DATASET_PATH,
    '--output', OUTPUT_DIR,
    '--model', MODEL_NAME,
    '--epochs', '1.0',
    '--per-device-train-batch-size', '1',
    '--gradient-accumulation-steps', '8',
    '--logging-steps', '5',
    '--save-steps', '50',
    '--min-score-gap', '0.05',
    '--report-to', 'none',
    '--run-name', 'aegisdesk-kaggle-dpo',
]
print('Running:', ' \n'.join(train_cmd[:3]), '...')
train_env = dict(os.environ)
train_env['WANDB_DISABLED'] = 'true'
subprocess.run(train_cmd, check=True, env=train_env)


## Next Step

After training, evaluate the saved adapter with the project benchmark script or your preferred AegisDesk eval flow.

If you want a stronger DPO run after this, the next upgrade is to harvest real AegisDesk win/fail trajectories and rebuild `training/data/aegisdesk_pref.jsonl` with project-heavy pairs.
